# Modeling Win Probability for a Marth Player Against Fox

In [1]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az

# Set random seed
seed = 6420
rng = np.random.default_rng(seed)

In [2]:
# Load data
data = pd.read_csv("data/fox_marth_competitive.csv")

# Append synthetic elo data for each character (1000 - 2400)
data["fox_elo"] = rng.integers(1000, 2400, size=len(data))
data["marth_elo"] = rng.integers(1000, 2400, size=len(data))

# Count wins for each character
num_matches = len(data)
num_marth_wins = (data["winning_character"] == "MARTH").sum()
num_fox_wins = num_matches - num_marth_wins
print(f"Total matches: {num_matches}")
print(f"Matches won by Marth: {num_marth_wins}")
print(f"Matches won by Fox: {num_fox_wins}")

data = data.head(10000)  # Limit to first 10000 rows for faster sampling

# Show the first few rows of the data
print(data.head())

Total matches: 270974
Matches won by Marth: 133582
Matches won by Fox: 137392
   id               stage winning_character  fox_elo  marth_elo
0   1        YOSHIS_STORY               FOX     2320       1533
1   2  FOUNTAIN_OF_DREAMS             MARTH     1265       1861
2   3   FINAL_DESTINATION               FOX     1173       1114
3   4   FINAL_DESTINATION             MARTH     1530       1910
4   5        YOSHIS_STORY               FOX     1269       1277


In [3]:
stage_idx, stages = pd.factorize(data["stage"])
print("Unique stages:", stages)
print("Stage indices:", stage_idx)
print("Stage labels:", stages.tolist())
print("Number of unique stages:", len(stages))

Unique stages: Index(['YOSHIS_STORY', 'FOUNTAIN_OF_DREAMS', 'FINAL_DESTINATION',
       'POKEMON_STADIUM', 'BATTLEFIELD', 'DREAMLAND'],
      dtype='str')
Stage indices: [0 1 2 ... 5 5 1]
Stage labels: ['YOSHIS_STORY', 'FOUNTAIN_OF_DREAMS', 'FINAL_DESTINATION', 'POKEMON_STADIUM', 'BATTLEFIELD', 'DREAMLAND']
Number of unique stages: 6


In [4]:
y = (data["winning_character"] == "MARTH").astype(int).values
y[:10]

array([0, 1, 0, 1, 0, 1, 0, 1, 1, 0])

In [5]:
# Model with just stage effects
with pm.Model(coords={"stage": stages}) as stage_only_model:

    # Stage effects
    gamma = pm.ZeroSumNormal("gamma", sigma=10, dims="stage")

    # Matchup effect
    alpha = pm.Normal("alpha", mu=0.0, sigma=10.0)

    # Linear predictor
    logit_p = gamma[stage_idx] + alpha

    # Likelihood
    p = pm.math.sigmoid(logit_p)
    pm.Deterministic("p_mean", p.mean())

    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)

    stage_only_trace = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=seed)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [gamma, alpha]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 6 seconds.


In [6]:
summary = az.summary(stage_only_trace, hdi_prob=0.95)
summary

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-0.284,0.020,-0.321,-0.243,0.000,0.000,5294.0,3306.0,1.0
gamma[YOSHIS_STORY],0.121,0.045,0.036,0.210,0.001,0.001,5943.0,2928.0,1.0
gamma[FOUNTAIN_OF_DREAMS],-0.019,0.046,-0.107,0.070,0.001,0.001,5386.0,3267.0,1.0
gamma[FINAL_DESTINATION],0.069,0.046,-0.019,0.163,0.001,0.001,6276.0,2805.0,1.0
gamma[POKEMON_STADIUM],-0.081,0.043,-0.162,0.004,0.001,0.001,5336.0,2909.0,1.0
gamma[BATTLEFIELD],-0.019,0.044,-0.108,0.060,0.001,0.001,6070.0,3172.0,1.0
gamma[DREAMLAND],-0.070,0.046,-0.161,0.017,0.001,0.001,5253.0,3418.0,1.0
p_mean,0.430,0.005,0.421,0.440,0.000,0.000,5328.0,3015.0,1.0


In [7]:
print(summary.to_latex(escape=True, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrr}
\toprule
 & mean & sd & hdi\_2.5\% & hdi\_97.5\% & mcse\_mean & mcse\_sd & ess\_bulk & ess\_tail & r\_hat \\
\midrule
alpha & -0.284 & 0.020 & -0.321 & -0.243 & 0.000 & 0.000 & 5294.000 & 3306.000 & 1.000 \\
gamma[YOSHIS\_STORY] & 0.121 & 0.045 & 0.036 & 0.210 & 0.001 & 0.001 & 5943.000 & 2928.000 & 1.000 \\
gamma[FOUNTAIN\_OF\_DREAMS] & -0.019 & 0.046 & -0.107 & 0.070 & 0.001 & 0.001 & 5386.000 & 3267.000 & 1.000 \\
gamma[FINAL\_DESTINATION] & 0.069 & 0.046 & -0.019 & 0.163 & 0.001 & 0.001 & 6276.000 & 2805.000 & 1.000 \\
gamma[POKEMON\_STADIUM] & -0.081 & 0.043 & -0.162 & 0.004 & 0.001 & 0.001 & 5336.000 & 2909.000 & 1.000 \\
gamma[BATTLEFIELD] & -0.019 & 0.044 & -0.108 & 0.060 & 0.001 & 0.001 & 6070.000 & 3172.000 & 1.000 \\
gamma[DREAMLAND] & -0.070 & 0.046 & -0.161 & 0.017 & 0.001 & 0.001 & 5253.000 & 3418.000 & 1.000 \\
p\_mean & 0.430 & 0.005 & 0.421 & 0.440 & 0.000 & 0.000 & 5328.000 & 3015.000 & 1.000 \\
\bottomrule
\end{tabular}



In [8]:
# Suppose you have the posterior mean for alpha
alpha_mean = summary.loc["alpha", "mean"]  # or use the correct column if different

# Convert to probability using invlogit
prob = 1 / (1 + np.exp(-alpha_mean))
print(f"Posterior mean probability for alpha: {prob:.3f}")

Posterior mean probability for alpha: 0.429


Testing with a lower confidence interval

In [9]:
summary = az.summary(stage_only_trace, hdi_prob=0.9)
summary

,mean,sd,hdi_5%,hdi_95%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-0.284,0.020,-0.315,-0.250,0.000,0.000,5294.0,3306.0,1.0
gamma[YOSHIS_STORY],0.121,0.045,0.050,0.196,0.001,0.001,5943.0,2928.0,1.0
gamma[FOUNTAIN_OF_DREAMS],-0.019,0.046,-0.093,0.058,0.001,0.001,5386.0,3267.0,1.0
gamma[FINAL_DESTINATION],0.069,0.046,-0.004,0.147,0.001,0.001,6276.0,2805.0,1.0
gamma[POKEMON_STADIUM],-0.081,0.043,-0.154,-0.013,0.001,0.001,5336.0,2909.0,1.0
gamma[BATTLEFIELD],-0.019,0.044,-0.093,0.049,0.001,0.001,6070.0,3172.0,1.0
gamma[DREAMLAND],-0.070,0.046,-0.147,0.004,0.001,0.001,5253.0,3418.0,1.0
p_mean,0.430,0.005,0.422,0.438,0.000,0.000,5328.0,3015.0,1.0


In [10]:
print(summary.to_latex(escape=True, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrr}
\toprule
 & mean & sd & hdi\_5\% & hdi\_95\% & mcse\_mean & mcse\_sd & ess\_bulk & ess\_tail & r\_hat \\
\midrule
alpha & -0.284 & 0.020 & -0.315 & -0.250 & 0.000 & 0.000 & 5294.000 & 3306.000 & 1.000 \\
gamma[YOSHIS\_STORY] & 0.121 & 0.045 & 0.050 & 0.196 & 0.001 & 0.001 & 5943.000 & 2928.000 & 1.000 \\
gamma[FOUNTAIN\_OF\_DREAMS] & -0.019 & 0.046 & -0.093 & 0.058 & 0.001 & 0.001 & 5386.000 & 3267.000 & 1.000 \\
gamma[FINAL\_DESTINATION] & 0.069 & 0.046 & -0.004 & 0.147 & 0.001 & 0.001 & 6276.000 & 2805.000 & 1.000 \\
gamma[POKEMON\_STADIUM] & -0.081 & 0.043 & -0.154 & -0.013 & 0.001 & 0.001 & 5336.000 & 2909.000 & 1.000 \\
gamma[BATTLEFIELD] & -0.019 & 0.044 & -0.093 & 0.049 & 0.001 & 0.001 & 6070.000 & 3172.000 & 1.000 \\
gamma[DREAMLAND] & -0.070 & 0.046 & -0.147 & 0.004 & 0.001 & 0.001 & 5253.000 & 3418.000 & 1.000 \\
p\_mean & 0.430 & 0.005 & 0.422 & 0.438 & 0.000 & 0.000 & 5328.000 & 3015.000 & 1.000 \\
\bottomrule
\end{tabular}



Introducing an "Elo" effect

In [11]:
# We normalize elo by a factor of 100, which is an approximate diffrence in elo for different slippi ranks
elo_diff = (data["marth_elo"].values - data["fox_elo"].values) / 100.0
print("Number of matches:", len(y))
print("Number of matches won by Marth:", y.sum())
print("Number of matches won by Fox:", len(y) - y.sum())

print("Marth winrate on Yoshi's Story:", np.sum((data["stage"] == "YOSHIS_STORY") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "YOSHIS_STORY"))
print("Marth winrate on Battlefield:", np.sum((data["stage"] == "BATTLEFIELD") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "BATTLEFIELD"))
print("Marth winrate on Final Destination:", np.sum((data["stage"] == "FINAL_DESTINATION") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "FINAL_DESTINATION"))
print("Marth winrate on Dreamland:", np.sum((data["stage"] == "DREAMLAND") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "DREAMLAND"))
print("Marth winrate on Fountain of Dreams:", np.sum((data["stage"] == "FOUNTAIN_OF_DREAMS") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "FOUNTAIN_OF_DREAMS"))
print("Marth winrate on Pokemon Stadium:", np.sum((data["stage"] == "POKEMON_STADIUM") & (data["winning_character"] == "MARTH")) / np.sum(data["stage"] == "POKEMON_STADIUM"))

with pm.Model(coords={"stage": stages}) as model:

    # Elo scaling
    beta = pm.Normal("beta", mu=1, sigma=5)

    # Global matchup (Marth vs Fox)
    alpha = pm.Normal("alpha", mu=0.0, sigma=10.0)

    # Stage effects
    gamma = pm.ZeroSumNormal("gamma", sigma=0.5, dims="stage")

    # Linear predictor
    logit_p = beta * elo_diff + alpha + gamma[stage_idx]

    # Likelihood
    p = pm.math.sigmoid(logit_p)
    pm.Deterministic("p_mean", p.mean())
    
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)

    trace = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=seed)

Number of matches: 10000
Number of matches won by Marth: 4298
Number of matches won by Fox: 5702
Marth winrate on Yoshi's Story: 0.45944412932501416
Marth winrate on Battlefield: 0.4249568717653824
Marth winrate on Final Destination: 0.4464968152866242
Marth winrate on Dreamland: 0.4126092384519351
Marth winrate on Fountain of Dreams: 0.42490613266583227
Marth winrate on Pokemon Stadium: 0.4097222222222222


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, alpha, gamma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 9 seconds.


In [12]:
az.summary(trace, var_names=["alpha", "beta", "gamma"], hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-0.283,0.020,-0.320,-0.243,0.000,0.000,6094.0,2821.0,1.0
beta,0.001,0.003,-0.006,0.008,0.000,0.000,6041.0,2782.0,1.0
gamma[YOSHIS_STORY],0.120,0.044,0.037,0.204,0.000,0.001,7817.0,3019.0,1.0
gamma[FOUNTAIN_OF_DREAMS],-0.019,0.046,-0.111,0.065,0.001,0.001,6525.0,3159.0,1.0
gamma[FINAL_DESTINATION],0.068,0.046,-0.021,0.156,0.001,0.001,7242.0,3334.0,1.0
gamma[POKEMON_STADIUM],-0.081,0.043,-0.166,0.001,0.001,0.001,5869.0,3162.0,1.0
gamma[BATTLEFIELD],-0.018,0.045,-0.107,0.067,0.001,0.001,7114.0,3130.0,1.0
gamma[DREAMLAND],-0.070,0.048,-0.165,0.019,0.001,0.001,6139.0,2948.0,1.0
